In [1]:
from __future__ import annotations
import pandas as pd
from tqdm import tqdm
from pathlib import Path
import argparse, sys, time, yaml
from src.etl.gpu import GpuETL
from src.etl.seq import SeqETL
from src.etl.slurm import SlurmETL
from src.model.train import run as train_model
from src.model.inference import run as inf_model

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
gpu = GpuETL()
# gpu.run()
gpu._build_traces()

In [ ]:
slurm = SlurmETL()
tasks = [("Reading and processing intermediate parquet", slurm.slurm_intermediate_parquet), 
         ("Plotting features", slurm.plot_features), 
         ("Creating correlation heatmap", slurm.correlation_heatmap)]

for desc, func in tqdm(tasks, desc="SlurmETL Tasks Progress", unit="task"):
    tqdm.write(f"Starting: {desc}")
    func()

In [6]:
SeqETL().run()

Loaded config: /scratch/aa9360/supercloud_power/configs/seqETL.yaml

── Phase 1: Load merged table ──
  Loaded merged features for 93,362 jobs
  Loaded jobs: 93,362

── Phase 2: Stratified split ──
  Train: 84,026  Test: 9,336 (no stratification)

── Phase 3: Norm stats fit (train only) ──
  Fitting stats on all 84,026 train jobs


  memory_used_MiB: log1p_mean=8.133958, log1p_std=2.836587, n=27,257,673,601
  power_draw_W: log1p_mean=4.322631, log1p_std=0.716482, n=27,257,673,601
  Wrote /scratch/aa9360/supercloud_power/data/p/norm_stats.npz

── Phase 4: Normalize parquets → NPY ──
Total jobs: 93362, Total batches: 5


Batch 1 completed


Batch 2 completed


Batch 3 completed


Batch 4 completed


Batch 5 completed
  NPY: 93,362 ok, 0 failed in 29.0s
  Successful: 93,362, Failed: 0

── Phase 5: Build chunk index ──
  train: 84,026 jobs → 26,660,682 chunks
  test: 9,336 jobs → 2,920,204 chunks
  Wrote /scratch/aa9360/supercloud_power/data/p/train_chunks.parquet (26,660,682 chunks)
  Wrote /scratch/aa9360/supercloud_power/data/p/train_jobs.parquet (84,026 jobs)
  Wrote /scratch/aa9360/supercloud_power/data/p/test_chunks.parquet (2,920,204 chunks)
  Wrote /scratch/aa9360/supercloud_power/data/p/test_jobs.parquet (9,336 jobs)

Total elapsed: 71.3s
Report: /scratch/aa9360/supercloud_power/data/p/seq-etl-report.txt


0

In [2]:
with open('configs/v5.yaml') as f:
    cfg = yaml.safe_load(f)

t0 = time.time()
rc = 0
print("Stage 1: Train")
rc = train_model(cfg)
if rc:
    print(f"Train returned {rc}")


print(f"\n{'=' * 70}\n Pipeline complete in {(time.time() - t0) / 60:.1f} min\n{'=' * 70}")


Stage 1: Train
Detected resources: {'cpu_count': 128, 'workers': 126, 'gpu_count': 2, 'gpu_name': 'NVIDIA H200', 'vram_gb': 150.110011392, 'bf16_supported': True}


W0505 23:28:32.095000 3472379 torch/multiprocessing/spawn.py:165] Terminating process 3474082 via signal SIGTERM


ProcessRaisedException: 

-- Process 0 terminated with the following error:
Traceback (most recent call last):
  File "/home/aa9360/.local/lib/python3.13/site-packages/torch/multiprocessing/spawn.py", line 87, in _wrap
    fn(i, *args)
    ~~^^^^^^^^^^
  File "/scratch/aa9360/supercloud_power/src/model/train.py", line 102, in train_one_rank
    _ddp_setup(rank, world_size)
    ~~~~~~~~~~^^^^^^^^^^^^^^^^^^
  File "/scratch/aa9360/supercloud_power/src/model/train.py", line 47, in _ddp_setup
    dist.init_process_group("nccl", rank=rank, world_size=world_size)
    ~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/aa9360/.local/lib/python3.13/site-packages/torch/distributed/c10d_logger.py", line 83, in wrapper
    return func(*args, **kwargs)
  File "/home/aa9360/.local/lib/python3.13/site-packages/torch/distributed/c10d_logger.py", line 97, in wrapper
    func_return = func(*args, **kwargs)
  File "/home/aa9360/.local/lib/python3.13/site-packages/torch/distri

In [ ]:
with open('configs/v5.yaml') as f:
    cfg = yaml.safe_load(f)

t0 = time.time()
print("Stage 2: Inference")
rc = inf_model(cfg)
if rc:
    print(f"Inference returned {rc}")
print(f"\n{'=' * 70}\n Pipeline complete in {(time.time() - t0) / 60:.1f} min\n{'=' * 70}")

